In [ ]:
module chebyshev_method

    using LinearAlgebra

    export chebyshev_D
    # Chebyshev compute D = differentiation matrix, x = Chebyshev grid

    function  chebyshev_D(N)

        if N==0
            D = 0;
            x = 1;
            return D, x
        else
            θ = range(0,length=N+1,stop=pi)
            x = reshape(-cos.(θ), N+1, 1)
            c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
            X = repeat(x, 1, N+1);
            dX = X - X';
            D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
            D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
            return D, x
        end
    end
end

In [ ]:
N = 99;      # Number of interior points
R = 2000;    # Reynolds number
ω  = 0.3;     # Input frequency
β  = 0

In [ ]:
using .chebyshev_method
using LinearAlgebra
using NonlinearEigenproblems
using SparseArrays
D,x=chebyshev_D(N)
D2=D^2
U=1 .-x.^2
dU=-2 .*x
ddU=-2*ones(N+1)
W=dW=zeros(N+1)
V=dV=ddV=zeros(N+1)
A=D2-(β^2)*I(N+1)
L0_1= im*A^2 + R*β*V.*A - R*ω.*A - R*β*ddV.*I(N+1) + im*ddU.*I(N+1) - im*dW.*A - im*U.*D^2
L0_2= 2*(V).*D + 2*dV.*I(N+1)
L0_3= im*R*β*dU.*I(N+1)
L0_4= im*A + R*β*V.*I(N+1) - R*ω*I(N+1) - im*W.*D - im*U.*I(N+1)
L1_1= -(1/R).*A + R*U.*A + im*β*V.*I(N+1) - im*ω*I(N+1) - R*ddU.*I(N+1) + (1/R)*W.*D + (1/R)*dW.*I(N+1)
L1_2= zeros(N+1,N+1)
L1_3= -1*im*R*dV.*I(N+1)
L1_4= R*U.*I(N+1)
L2_1= -2*im*A + ω*R*I(N+1) - β*R*V.*I(N+1) + im*U.*I(N+1) + im*W.*D + im*dW.*I(N+1)
L2_2= L1_2
L2_3= L1_2
L2_4= -im*I(N+1)
L3_1= (1/R)*I(N+1) - R*U.*I(N+1)
L3_2= L3_3=L3_4=L1_2
L4_1= im*I(N+1)
L4_2= L4_3=L4_4=L1_2
L2_4= Matrix{ComplexF64}(L2_4)
z = zeros(N-1,N-1)
c = I(N-1)
c1=(-im*W.*A)
c2=(2*(V).*I(N+1))
c1=c1[2:N,2:N]
c2=c2[2:N,2:N]
L0_1=L0_1[2:N,2:N];L0_2=L0_2[2:N,2:N];L0_3=L0_3[2:N,2:N];L0_4=L0_4[2:N,2:N];
L1_1=L1_1[2:N,2:N];L1_2=L1_2[2:N,2:N];L1_3=L1_3[2:N,2:N];L1_4=L1_4[2:N,2:N];
L2_1=L2_1[2:N,2:N];L2_2=L2_2[2:N,2:N];L2_3=L2_3[2:N,2:N];L2_4=L2_4[2:N,2:N];
L3_1=L3_1[2:N,2:N];L3_2=L3_2[2:N,2:N];L3_3=L3_3[2:N,2:N];L3_4=L3_4[2:N,2:N];
L4_1=L4_1[2:N,2:N];L4_2=L4_2[2:N,2:N];L4_3=L4_3[2:N,2:N];L4_4=L4_4[2:N,2:N]
A0=[L0_1 L0_2 c1;L0_3 L0_4 c2;D[2:N,2:N] z -1*c]
A1=[L1_1 L1_2 z;L1_3 L1_4 z;z z z]
A2=[L2_1 L2_2 z;L2_3 L2_4 z;z z z]
A3=[L3_1 L3_2 z;L3_3 L3_4 z;z z z]
A4=[L4_1 L4_2 z;L4_3 L4_4 z;z z z]


In [ ]:
A0=sparse(A0);
A1=sparse(A1);
A2=sparse(A2);
A3=sparse(A3);
A4=sparse(A4);
nep = PEP([A0,A1,A2,A3,A4]); #Create a PEP object

In [ ]:
norm.(get_Av(nep))

In [ ]:
sc=100
nep1 = shift_and_scale(nep,scale=sc);
mult_scale = norm(nep1.A[end]);
nep2 = PEP(nep1.A ./ mult_scale);
norm.(get_Av(nep2))
1

In [ ]:
λ1,v1 = iar(nep2,neigs=3,v=ones(size(nep,1)),maxit=100);
λ2,v2 = iar(nep2,neigs=3,v=ones(size(nep,1)),maxit=100);
λtotal = [λ1;λ2]

In [ ]:
λ_orig = sc*λtotal
filter(x->imag(x)<0,λ1)
λ_orig

In [ ]:
using Plots
plot(real.(λ_orig),imag.(λ_orig),seriestype=:scatter)